# 🚀 OpenCloud-SRE: Autonomous GRPO Training
This notebook contains the complete training pipeline for the OpenCloud-SRE agent using **Unsloth** and **GRPO** (Group Relative Policy Optimization).

### ⚠️ Important: Runtime Setup
To run this notebook, you **MUST** use a GPU runtime:
1. Go to **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (Standard) or higher.

### 🔗 Links
- **GitHub:** [https://github.com/hitendras510/OpenCloud-SRE](https://github.com/hitendras510/OpenCloud-SRE)
- **Hugging Face:** [hitendras510/OpenCloud-SRE-GRPO](https://huggingface.co/spaces/hitendras510/OpenCloud-SRE)

In [ ]:
# 1. Install Dependencies
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir trl transformers accelerate datasets bitsandbytes wandb matplotlib

In [ ]:
# 2. GPU Verification
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not found! Please switch to a T4 GPU runtime in Colab settings.")
print(f"Connected to GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Simulation Environment Setup
from dataclasses import dataclass
import torch

METRIC_MIN, METRIC_MAX = 0.0, 100.0
NOMINAL_STATE = [20.0, 30.0, 90.0]

@dataclass
class CloudStateTensor:
    traffic_load: float = 20.0
    database_temperature: float = 30.0
    network_health: float = 90.0

    def apply_delta(self, delta: torch.Tensor) -> 'CloudStateTensor':
        raw = torch.tensor([self.traffic_load, self.database_temperature, self.network_health]) + delta
        clamped = torch.clamp(raw, METRIC_MIN, METRIC_MAX).tolist()
        return CloudStateTensor(*clamped)

    def slo_score(self) -> float:
        nominal = torch.tensor(NOMINAL_STATE)
        current = torch.tensor([self.traffic_load, self.database_temperature, self.network_health])
        dist = torch.norm(current - nominal).item()
        max_dist = torch.norm(torch.tensor([100.0, 100.0, 0.0]) - nominal).item()
        return max(0.0, 1.0 - dist / max_dist)

class OpenCloudEnv:
    def __init__(self):
        self.reset()
    def reset(self):
        self.state = CloudStateTensor(98.0, 95.0, 5.0) # Start crashed
        return self.state
    def step(self, action):
        deltas = {"scale_out": (-15, -10, 15), "noop": (3, 2, -3)}
        base = deltas.get(action, (-5, -5, 5))
        self.state = self.state.apply_delta(torch.tensor(base, dtype=torch.float32))
        return self.state, self.state.slo_score()

In [ ]:
# 4. GRPO Configuration & Reward Functions
def reward_fn(responses, states):
    rewards = []
    for response, state in zip(responses, states):
        # Reward high SLO scores and valid JSON formatting
        score = state.slo_score()
        reward = score * 10.0
        if "action" in response.lower():
            reward += 2.0
        rewards.append(reward)
    return rewards

In [ ]:
# 5. Load Model & Start Training
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

print("Model loaded! Agent is ready for GRPO optimization.")